# Parte 1: Conversión de archivos

## Imports

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Configuración de rutas y columnas

In [2]:
# Rutas de los archivos originales (ajustar si es necesario)
RUTA_CASEN = "casen/casen_2024.dta"
RUTA_PROV_COMUNA = "casen/casen_2024_provincia_comuna.dta"

# Rutas de salida
RUTA_CSV_CASEN = "casen/casen_2024.csv"
RUTA_CSV_PROV_COMUNA = "casen/casen_2024_provincia_comuna.csv"

# Columnas seleccionadas para el proyecto (17 en total)
# según el modelo relacional definido en el Hito 1
COLUMNAS = [
    # Llaves y expansión
    "folio", "id_persona", "expr",
    # Demografía y geografía
    # Nota: 'provincia' y 'comuna' NO están en el archivo principal;
    # vienen en el archivo auxiliar casen_2024_provincia_comuna.dta
    "sexo", "edad", "region", "area",
    # Educación
    "e6a", "cinef13_area", "cinef13_subarea",
    # Trabajo
    "activ", "o15", "rama1", "rama4",
    # Ingresos
    "yoprcor",
]

len(COLUMNAS)

15

## Lectura del archivo principal

In [3]:
df_casen = pd.read_stata(
    RUTA_CASEN,
    columns=COLUMNAS,
    convert_categoricals=False,
)

print(f"Filas: {df_casen.shape[0]:,}")
print(f"Columnas: {df_casen.shape[1]}")
df_casen.head()

Filas: 218,367
Columnas: 15


,folio,id_persona,expr,sexo,edad,region,area,e6a,cinef13_area,cinef13_subarea,activ,o15,rama1,rama4,yoprcor
0,100020301,1,264,1,62,13,1,9,NaN,NaN,3.0,NaN,NaN,NaN,NaN
1,100020301,2,264,2,92,13,1,7,NaN,NaN,3.0,NaN,NaN,NaN,NaN
2,100020401,1,295,1,61,13,1,7,NaN,NaN,1.0,5.0,7.0,4805.0,1200000.0
3,100020401,2,295,2,61,13,1,7,NaN,NaN,1.0,5.0,12.0,6801.0,500000.0
4,100020401,3,295,1,32,13,1,12,5.0,5.0,1.0,5.0,7.0,4808.0,700000.0


## Lectura del archivo auxiliar de provincia y comuna

In [4]:
df_prov_comuna = pd.read_stata(
    RUTA_PROV_COMUNA,
    convert_categoricals=False,
)

print(f"Filas: {df_prov_comuna.shape[0]:,}")
print(f"Columnas: {df_prov_comuna.shape[1]}")
print("Columnas disponibles:", list(df_prov_comuna.columns))
df_prov_comuna.head()

Filas: 218,367
Columnas: 6
Columnas disponibles: ['folio', 'id_persona', 'expp', 'expc', 'provincia', 'comuna']


,folio,id_persona,expp,expc,provincia,comuna
0,100020301,1,285,371,131,13127
1,100020301,2,285,371,131,13127
2,100020401,1,286,371,131,13127
3,100020401,2,286,371,131,13127
4,100020401,3,286,371,131,13127


## Merge del archivo principal con el auxiliar

In [5]:
# Ajustar las claves del merge según lo que se vio en la Celda 4
CLAVES_MERGE = ["folio", "id_persona"]  # cambiar a ["folio"] si corresponde

# Columnas del auxiliar que queremos incorporar:
# - provincia, comuna: códigos geográficos
# - expp, expc: factores de expansión provincial y comunal
COLS_AUX = ["provincia", "comuna", "expp", "expc"]

cols_a_traer = CLAVES_MERGE + [c for c in COLS_AUX if c in df_prov_comuna.columns]
df_prov_comuna_min = df_prov_comuna[cols_a_traer]

df_casen = df_casen.merge(
    df_prov_comuna_min,
    on=CLAVES_MERGE,
    how="left",
    validate="one_to_one",  # si falla, probar "many_to_one"
)

print(f"Filas tras el merge: {df_casen.shape[0]:,}")
print(f"Columnas tras el merge: {df_casen.shape[1]}")
for col in ["provincia", "comuna", "expp", "expc"]:
    if col in df_casen.columns:
        print(f"Nulos en {col}: {df_casen[col].isna().sum()}")
df_casen.head()

Filas tras el merge: 218,367
Columnas tras el merge: 19
Nulos en provincia: 0
Nulos en comuna: 0
Nulos en expp: 0
Nulos en expc: 0


,folio,id_persona,expr,sexo,edad,region,area,e6a,cinef13_area,cinef13_subarea,activ,o15,rama1,rama4,yoprcor,provincia,comuna,expp,expc
0,100020301,1,264,1,62,13,1,9,NaN,NaN,3.0,NaN,NaN,NaN,NaN,131,13127,285,371
1,100020301,2,264,2,92,13,1,7,NaN,NaN,3.0,NaN,NaN,NaN,NaN,131,13127,285,371
2,100020401,1,295,1,61,13,1,7,NaN,NaN,1.0,5.0,7.0,4805.0,1200000.0,131,13127,286,371
3,100020401,2,295,2,61,13,1,7,NaN,NaN,1.0,5.0,12.0,6801.0,500000.0,131,13127,286,371
4,100020401,3,295,1,32,13,1,12,5.0,5.0,1.0,5.0,7.0,4808.0,700000.0,131,13127,286,371


## Inspección rápida de tipos y nulos

In [6]:
resumen = pd.DataFrame({
    "dtype": df_casen.dtypes,
    "no_nulos": df_casen.notna().sum(),
    "nulos": df_casen.isna().sum(),
    "% nulo": (df_casen.isna().mean() * 100).round(1),
    "unicos": df_casen.nunique(dropna=True),
})
resumen

,dtype,no_nulos,nulos,% nulo,unicos
folio,int32,218367,0,0.0,78654
id_persona,int8,218367,0,0.0,22
expr,int16,218367,0,0.0,687
sexo,int8,218367,0,0.0,2
edad,int16,218367,0,0.0,107
region,int8,218367,0,0.0,16
area,int8,218367,0,0.0,2
e6a,int8,218367,0,0.0,15
cinef13_area,float64,61070,157297,72.0,13
cinef13_subarea,float64,61070,157297,72.0,30


## Exportar a CSV

In [7]:
df_casen.to_csv(RUTA_CSV_CASEN, index=False)
df_prov_comuna.to_csv(RUTA_CSV_PROV_COMUNA, index=False)

print(f"Guardado: {RUTA_CSV_CASEN}")
print(f"Guardado: {RUTA_CSV_PROV_COMUNA}")

Guardado: casen/casen_2024.csv
Guardado: casen/casen_2024_provincia_comuna.csv


## Verificación de los CSV generados

In [8]:
df_check = pd.read_csv(RUTA_CSV_CASEN)
print(f"Filas leídas: {df_check.shape[0]:,}")
print(f"Columnas leídas: {df_check.shape[1]}")
assert df_check.shape == df_casen.shape, "Diferencia de dimensiones entre .dta y .csv"
print("OK: el CSV coincide en dimensiones con el .dta original")

df_check.head()

Filas leídas: 218,367
Columnas leídas: 19
OK: el CSV coincide en dimensiones con el .dta original


,folio,id_persona,expr,sexo,edad,region,area,e6a,cinef13_area,cinef13_subarea,activ,o15,rama1,rama4,yoprcor,provincia,comuna,expp,expc
0,100020301,1,264,1,62,13,1,9,NaN,NaN,3.0,NaN,NaN,NaN,NaN,131,13127,285,371
1,100020301,2,264,2,92,13,1,7,NaN,NaN,3.0,NaN,NaN,NaN,NaN,131,13127,285,371
2,100020401,1,295,1,61,13,1,7,NaN,NaN,1.0,5.0,7.0,4805.0,1200000.0,131,13127,286,371
3,100020401,2,295,2,61,13,1,7,NaN,NaN,1.0,5.0,12.0,6801.0,500000.0,131,13127,286,371
4,100020401,3,295,1,32,13,1,12,5.0,5.0,1.0,5.0,7.0,4808.0,700000.0,131,13127,286,371


# Parte 2: Construcción de tablas según el modelo relacional

## Limpieza de códigos especiales

In [9]:
COLS_CATEGORICAS = [
    "e6a", "cinef13_area", "cinef13_subarea",
    "activ", "o15", "rama1", "rama4",
]

CODIGOS_ESPECIALES = [-99, -88, -66]

print("Conteo de códigos especiales antes de limpiar:")
for col in COLS_CATEGORICAS:
    presentes = df_casen[col].isin(CODIGOS_ESPECIALES).sum()
    if presentes > 0:
        detalle = df_casen.loc[df_casen[col].isin(CODIGOS_ESPECIALES), col].value_counts()
        print(f"  {col}: {presentes} filas  ->  {dict(detalle)}")

# Reemplazar por NaN
for col in COLS_CATEGORICAS:
    df_casen.loc[df_casen[col].isin(CODIGOS_ESPECIALES), col] = np.nan

# Verificación
print("\nDespués de limpiar:")
for col in COLS_CATEGORICAS:
    quedan = df_casen[col].isin(CODIGOS_ESPECIALES).sum()
    assert quedan == 0, f"Quedan {quedan} códigos especiales en {col}"
print("OK: todos los códigos especiales reemplazados por NaN")

Conteo de códigos especiales antes de limpiar:
  cinef13_area: 156 filas  ->  {-66.0: np.int64(92), -88.0: np.int64(52), -99.0: np.int64(12)}
  cinef13_subarea: 156 filas  ->  {-66.0: np.int64(92), -88.0: np.int64(52), -99.0: np.int64(12)}
  rama1: 40 filas  ->  {-66.0: np.int64(26), -99.0: np.int64(10), -88.0: np.int64(4)}
  rama4: 40 filas  ->  {-66.0: np.int64(26), -99.0: np.int64(10), -88.0: np.int64(4)}

Después de limpiar:
OK: todos los códigos especiales reemplazados por NaN


## Carga de libro de codigos

In [10]:
RUTA_LIBRO = "casen/Libro_de_codigos_Casen_2024.xlsx"

def cargar_libro_completo(ruta):
    """Carga todas las hojas del libro como dict {hoja: DataFrame}, con ffill en Nombre variable."""
    xls = pd.ExcelFile(ruta)
    libro = {}
    for hoja in xls.sheet_names:
        try:
            df = pd.read_excel(ruta, sheet_name=hoja, header=2)
            if "Nombre variable" in df.columns:
                df["Nombre variable"] = df["Nombre variable"].ffill()
                libro[hoja] = df
        except Exception:
            pass  # hojas sin formato de libro de códigos (Índice, Ficha téc, etc.)
    return libro

LIBRO = cargar_libro_completo(RUTA_LIBRO)
print(f"Hojas con formato de libro de códigos: {list(LIBRO.keys())}")

Hojas con formato de libro de códigos: ['HdR', 'H', 'E', 'O', 'Y', 'S', 'R', 'V', 'Osig', 'Ing Cepal', 'Ing MDSF', 'PM', 'Var MDSF']


In [11]:
def parsear_variable(libro, nombre_var):
    """Devuelve un DataFrame con columnas (codigo, descripcion) para la variable indicada."""
    for hoja, df in libro.items():
        if nombre_var in df["Nombre variable"].values:
            subset = df[df["Nombre variable"] == nombre_var]
            out = (
                subset[["Valores", "Etiquetas de valores"]]
                .rename(columns={"Valores": "codigo", "Etiquetas de valores": "descripcion"})
                .copy()
            )
            # Quedarse solo con códigos numéricos (descarta filas tipo "Texto", "rango: ...")
            out["codigo"] = pd.to_numeric(out["codigo"], errors="coerce")
            out = out.dropna(subset=["codigo"]).copy()
            out["codigo"] = out["codigo"].astype(int)
            # Limpiar prefijo "N. " o "-N. " de la etiqueta
            out["descripcion"] = (
                out["descripcion"].astype(str)
                .str.replace(r"^-?\d+\.\s*", "", regex=True)
                .str.strip()
            )
            print(f"  '{nombre_var}' encontrada en hoja '{hoja}' ({len(out)} códigos)")
            return out.reset_index(drop=True)
    raise ValueError(f"Variable '{nombre_var}' no encontrada en el libro de códigos")

# Test rápido
_test = parsear_variable(LIBRO, "e6a")
_test.head()

  'e6a' encontrada en hoja 'E' (15 códigos)


,codigo,descripcion
0,1,Nunca asistió
1,2,Sala cuna
2,3,Jardín Infantil (Medio menor y Medio mayor)
3,4,Prekínder / Kínder (Transición menor y Transic...
4,5,Educación Especial (Diferencial)


In [12]:
df_cut_raw = pd.read_excel(RUTA_LIBRO, sheet_name="Anexo4", header=2)

# Las primeras dos columnas suelen estar vacías (índice y separador del .xlsx)
# y los nombres de columna pueden traer espacios extra. Nos quedamos con las útiles.
df_cut = df_cut_raw[[
    "Código Región", "Nombre Región",
    "Código Provincia", "Nombre Provincia",
    "Código Comuna", "Nombre Comuna",
]].rename(columns={
    "Código Región":    "cod_region",
    "Nombre Región":    "nom_region",
    "Código Provincia": "cod_provincia",
    "Nombre Provincia": "nom_provincia",
    "Código Comuna":    "cod_comuna",
    "Nombre Comuna":    "nom_comuna",
}).dropna(subset=["cod_comuna"]).copy()

# Códigos a int (en el .xlsx vienen como "011", "021"...; en CASEN como 11, 21)
for col in ["cod_region", "cod_provincia", "cod_comuna"]:
    df_cut[col] = df_cut[col].astype(int)

# Limpieza del prefijo "Región de " / "Región del "
df_cut["nom_region"] = (
    df_cut["nom_region"]
    .str.replace(r"^Región d[el]+\s+", "", regex=True)
    .str.strip()
)

print(f"Filas del CUT: {len(df_cut)}")
df_cut.head()

Filas del CUT: 346


,cod_region,nom_region,cod_provincia,nom_provincia,cod_comuna,nom_comuna
0,1,Tarapacá,11,Iquique,1101,Iquique
1,1,Tarapacá,11,Iquique,1107,Alto Hospicio
2,1,Tarapacá,14,Tamarugal,1401,Pozo Almonte
3,1,Tarapacá,14,Tamarugal,1402,Camiña
4,1,Tarapacá,14,Tamarugal,1403,Colchane


## Tabla Region

In [13]:
df_region = (
    df_cut[["cod_region", "nom_region"]]
    .drop_duplicates()
    .sort_values("cod_region")
    .rename(columns={"nom_region": "nombre"})
    .reset_index(drop=True)
)

# Validación: todas las regiones del sample CASEN deben estar en el CUT
faltantes = set(df_casen["region"].unique()) - set(df_region["cod_region"])
assert not faltantes, f"Regiones en CASEN no encontradas en CUT: {faltantes}"

print(f"Regiones: {len(df_region)}")
df_region

Regiones: 16


,cod_region,nombre
0,1,Tarapacá
1,2,Antofagasta
2,3,Atacama
3,4,Coquimbo
4,5,Valparaíso
5,6,Libertador Gral. Bernardo O'Higgins
6,7,Maule
7,8,Biobío
8,9,La Araucanía
9,10,Los Lagos


## Tabla Provincia

In [14]:
df_provincia = (
    df_cut[["cod_provincia", "cod_region", "nom_provincia"]]
    .drop_duplicates()
    .sort_values(["cod_region", "cod_provincia"])
    .rename(columns={"nom_provincia": "nombre"})
    .reset_index(drop=True)
)

assert df_provincia["cod_provincia"].is_unique, "Una provincia mapea a múltiples regiones"
faltantes = set(df_casen["provincia"].unique()) - set(df_provincia["cod_provincia"])
assert not faltantes, f"Provincias en CASEN no encontradas en CUT: {faltantes}"

print(f"Provincias: {len(df_provincia)}")
df_provincia.head(10)

Provincias: 56


,cod_provincia,cod_region,nombre
0,11,1,Iquique
1,14,1,Tamarugal
2,21,2,Antofagasta
3,22,2,El Loa
4,23,2,Tocopilla
5,31,3,Copiapó
6,32,3,Chañaral
7,33,3,Huasco
8,41,4,Elqui
9,42,4,Choapa


## Tabla Comuna

In [15]:
df_comuna = (
    df_cut[["cod_comuna", "cod_provincia", "nom_comuna"]]
    .drop_duplicates()
    .sort_values(["cod_provincia", "cod_comuna"])
    .rename(columns={"nom_comuna": "nombre"})
    .reset_index(drop=True)
)

assert df_comuna["cod_comuna"].is_unique, "Una comuna mapea a múltiples provincias"
faltantes = set(df_casen["comuna"].unique()) - set(df_comuna["cod_comuna"])
assert not faltantes, f"Comunas en CASEN no encontradas en CUT: {sorted(faltantes)[:10]}"

print(f"Comunas: {len(df_comuna)}")
df_comuna.head(10)

Comunas: 346


,cod_comuna,cod_provincia,nombre
0,1101,11,Iquique
1,1107,11,Alto Hospicio
2,1401,14,Pozo Almonte
3,1402,14,Camiña
4,1403,14,Colchane
5,1404,14,Huara
6,1405,14,Pica
7,2101,21,Antofagasta
8,2102,21,Mejillones
9,2103,21,Sierra Gorda


## Tabla Nivel_educacional

In [16]:
df_nivel_educ = parsear_variable(LIBRO, "e6a")

# Validación: todos los códigos de e6a en CASEN deben estar en la tabla
faltantes = set(df_casen["e6a"].dropna().unique()) - set(df_nivel_educ["codigo"])
assert not faltantes, f"Códigos de e6a no encontrados en libro: {faltantes}"

print(f"Niveles educacionales: {len(df_nivel_educ)}")
df_nivel_educ

  'e6a' encontrada en hoja 'E' (15 códigos)
Niveles educacionales: 15


,codigo,descripcion
0,1,Nunca asistió
1,2,Sala cuna
2,3,Jardín Infantil (Medio menor y Medio mayor)
3,4,Prekínder / Kínder (Transición menor y Transic...
4,5,Educación Especial (Diferencial)
5,6,Primaria o Preparatoria (Sistema antiguo)
6,7,Educación Básica
7,8,Humanidades (Sistema Antiguo)
8,9,Educación Media Científico-Humanista
9,10,"Técnica, Comercial, Industrial o Normalista (S..."


## Parseo del Anexo 3 (CINE-F)

In [17]:
df_anexo3 = pd.read_excel(RUTA_LIBRO, sheet_name="Anexo3", header=None)

# Localizar las filas donde empieza cada sub-tabla buscando los títulos
# Las dos sub-tablas: "Anexo 3.1" (campo amplio) y "Anexo 3.2" (campo específico)
def buscar_inicio(df, texto):
    """Devuelve el índice de la primera fila cuya columna 2 contenga el texto."""
    for i, valor in df[2].items():
        if isinstance(valor, str) and texto in valor:
            return i
    raise ValueError(f"No se encontró '{texto}' en el anexo")

idx_31 = buscar_inicio(df_anexo3, "Anexo 3.1")
idx_32 = buscar_inicio(df_anexo3, "Anexo 3.2")
print(f"Anexo 3.1 (campo amplio)     empieza en fila {idx_31}")
print(f"Anexo 3.2 (campo específico) empieza en fila {idx_32}")

Anexo 3.1 (campo amplio)     empieza en fila 2
Anexo 3.2 (campo específico) empieza en fila 16


In [18]:
# Anexo 3.1: campo amplio (cinef13_area)
# Estructura observada: la fila siguiente al título tiene "N°" y "Glosa";
# las filas con datos van desde idx_31 + 2 hasta idx_32 - 1
df_cinef_amplio = (
    df_anexo3.iloc[idx_31 + 2 : idx_32, [2, 3]]
    .rename(columns={2: "codigo", 3: "descripcion"})
    .dropna(subset=["codigo"])
    .copy()
)
df_cinef_amplio["codigo"] = pd.to_numeric(df_cinef_amplio["codigo"], errors="coerce")
df_cinef_amplio = df_cinef_amplio.dropna(subset=["codigo"])
df_cinef_amplio["codigo"] = df_cinef_amplio["codigo"].astype(int)
df_cinef_amplio = df_cinef_amplio.reset_index(drop=True)

print(f"Campos amplios (Anexo 3.1): {len(df_cinef_amplio)}")
df_cinef_amplio

Campos amplios (Anexo 3.1): 11


,codigo,descripcion
0,1,Salud y Bienestar
1,2,"Ingeniería, Industria y Construcción"
2,3,Educación
3,4,Servicios
4,5,Administración de Empresas y Derecho
5,6,"Ciencias Sociales, Periodismo e Información"
6,7,"Ciencias naturales, matemáticas y estadística"
7,8,"Agricultura, Silvicultura, Pesca y Veterinaria"
8,9,Tecnología de la Información y la Comunicación...
9,10,Artes y Humanidades


In [19]:
# Anexo 3.2: campo específico (cinef13_subarea)
# Va desde idx_32 + 2 hasta el final del anexo
df_cinef_especifico = (
    df_anexo3.iloc[idx_32 + 2 :, [2, 3]]
    .rename(columns={2: "codigo", 3: "descripcion"})
    .dropna(subset=["codigo"])
    .copy()
)
df_cinef_especifico["codigo"] = pd.to_numeric(df_cinef_especifico["codigo"], errors="coerce")
df_cinef_especifico = df_cinef_especifico.dropna(subset=["codigo"])
df_cinef_especifico["codigo"] = df_cinef_especifico["codigo"].astype(int)
df_cinef_especifico = df_cinef_especifico.reset_index(drop=True)

print(f"Campos específicos (Anexo 3.2): {len(df_cinef_especifico)}")
df_cinef_especifico.head(10)

Campos específicos (Anexo 3.2): 28


,codigo,descripcion
0,1,Salud
1,2,Ingeniería y Profesiones Afines
2,3,Educación
3,4,Servicios personales
4,5,Educación Comercial y Administración
5,6,Periodismo e Información
6,7,Ciencias Sociales y del Comportamiento
7,8,Ciencias Físicas
8,9,Derecho
9,10,Servicios de Transportes


## Tabla Campo_estudio

In [20]:
df_campo = df_cinef_amplio.copy()

faltantes = set(df_casen["cinef13_area"].dropna().astype(int).unique()) - set(df_campo["codigo"])
assert not faltantes, f"Códigos de cinef13_area no encontrados: {faltantes}"

print(f"Campos de estudio: {len(df_campo)}")
df_campo

Campos de estudio: 11


,codigo,descripcion
0,1,Salud y Bienestar
1,2,"Ingeniería, Industria y Construcción"
2,3,Educación
3,4,Servicios
4,5,Administración de Empresas y Derecho
5,6,"Ciencias Sociales, Periodismo e Información"
6,7,"Ciencias naturales, matemáticas y estadística"
7,8,"Agricultura, Silvicultura, Pesca y Veterinaria"
8,9,Tecnología de la Información y la Comunicación...
9,10,Artes y Humanidades


## Tabla Subcampo_estudio

In [21]:
df_jerarquia = (
    df_casen[["cinef13_subarea", "cinef13_area"]]
    .dropna()
    .drop_duplicates()
    .astype(int)
    .rename(columns={"cinef13_subarea": "codigo", "cinef13_area": "codigo_campo"})
)

df_subcampo = (
    df_jerarquia
    .merge(df_cinef_especifico, on="codigo", how="left")
    .sort_values(["codigo_campo", "codigo"])
    .reset_index(drop=True)
)

assert df_subcampo["codigo"].is_unique, "Un subcampo mapea a múltiples campos en los datos"
sin_desc = df_subcampo[df_subcampo["descripcion"].isna()]
if len(sin_desc) > 0:
    print("Subcampos sin descripción (revisar parser del Anexo 3.2):")
    print(sin_desc)
assert df_subcampo["descripcion"].notna().all(), "Faltan descripciones de subcampos"

print(f"Subcampos de estudio: {len(df_subcampo)}")
df_subcampo.head(10)

Subcampos de estudio: 27


,codigo,codigo_campo,descripcion
0,1,1,Salud
1,15,1,Bienestar
2,2,2,Ingeniería y Profesiones Afines
3,11,2,Arquitectura y Construcción
4,23,2,Industria y Producción
5,3,3,Educación
6,4,4,Servicios personales
7,10,4,Servicios de Transportes
8,19,4,Servicios de Seguridad
9,22,4,Servicios de Higiene y Salud Ocupacional


## Tabla Categoria_ocupacional

In [22]:
df_cat_ocup = parsear_variable(LIBRO, "o15")

faltantes = set(df_casen["o15"].dropna().astype(int).unique()) - set(df_cat_ocup["codigo"])
assert not faltantes, f"Códigos de o15 no encontrados: {faltantes}"

print(f"Categorías ocupacionales: {len(df_cat_ocup)}")
df_cat_ocup

  'o15' encontrada en hoja 'O' (9 códigos)
Categorías ocupacionales: 9


,codigo,descripcion
0,1,Patrón(a) o empleador(a)
1,2,Trabajador(a) por cuenta propia
2,3,Empleado(a) u obrero(a) del sector público (Go...
3,4,Empleado(a) u obrero(a) de empresas públicas
4,5,Empleado(a) u obrero(a) del sector privado
5,6,Servicio doméstico puertas adentro
6,7,Servicio doméstico puertas afuera
7,8,FF.AA. y del Orden
8,9,Familiar no remunerado


## Parseo del Anexo 2 (CAENES)

In [23]:
df_anexo2 = pd.read_excel(RUTA_LIBRO, sheet_name="Anexo2", header=None)

# Las filas útiles empiezan después del header (filas 0-2 son títulos)
df_anexo2_datos = df_anexo2.iloc[3:].copy().reset_index(drop=True)
df_anexo2_datos.columns = ["c0", "seccion", "clase", "glosa"]

secciones = []   # {cod_rama, letra, descripcion}
clases = []      # {codigo, cod_rama, descripcion}

contador_seccion = 0
seccion_actual = None

for _, fila in df_anexo2_datos.iterrows():
    val_seccion = fila["seccion"]
    val_clase = fila["clase"]
    val_glosa = fila["glosa"]

    if isinstance(val_seccion, str) and val_seccion.strip():
        # Nueva sección: la descripción está en la columna 'clase'
        contador_seccion += 1
        seccion_actual = contador_seccion
        # Tomar la primera columna no vacía entre clase y glosa como descripción
        if isinstance(val_clase, str) and val_clase.strip():
            descripcion_seccion = val_clase.strip()
        elif isinstance(val_glosa, str) and val_glosa.strip():
            descripcion_seccion = val_glosa.strip()
        else:
            descripcion_seccion = None
        secciones.append({
            "cod_rama": contador_seccion,
            "letra": val_seccion.strip(),
            "descripcion": descripcion_seccion,
        })
    elif pd.notna(val_clase) and seccion_actual is not None:
        # Clase dentro de la sección actual: la descripción está en 'glosa'
        try:
            cod_clase = int(val_clase)
        except (ValueError, TypeError):
            continue
        clases.append({
            "codigo": cod_clase,
            "cod_rama": seccion_actual,
            "descripcion": val_glosa.strip() if isinstance(val_glosa, str) else None,
        })

df_caenes_seccion = pd.DataFrame(secciones)
df_caenes_clase = pd.DataFrame(clases)

# El Anexo 2 del libro oficial tiene al menos una clase duplicada (caso conocido:
# código 2701 aparece dos veces con la misma descripción y sección). Deduplicamos
# para evitar problemas de unicidad aguas abajo.
duplicados = df_caenes_clase[df_caenes_clase["codigo"].duplicated(keep=False)]
if len(duplicados) > 0:
    print(f"⚠ Clases duplicadas en Anexo 2 ({len(duplicados)} filas):")
    print(duplicados.to_string())
    print()
df_caenes_clase = df_caenes_clase.drop_duplicates(subset="codigo").reset_index(drop=True)

# Validación: todas las secciones y clases deben tener descripción
assert df_caenes_seccion["descripcion"].notna().all(), "Hay secciones sin descripción"
assert df_caenes_clase["descripcion"].notna().all(), "Hay clases sin descripción"
assert df_caenes_clase["codigo"].is_unique, "Quedan clases CAENES duplicadas tras deduplicar"

print(f"Secciones CAENES (rama1): {len(df_caenes_seccion)}")
print(f"Clases CAENES (rama4):    {len(df_caenes_clase)}")
print("\nSecciones:")
print(df_caenes_seccion)

⚠ Clases duplicadas en Anexo 2 (2 filas):
    codigo  cod_rama                                                                                    descripcion
32    2701         3  Fabricación de productos informáticos, electrónicos, ópticos, equipos eléctricos y maquinaria
33    2701         3  Fabricación de productos informáticos, electrónicos, ópticos, equipos eléctricos y maquinaria

Secciones CAENES (rama1): 21
Clases CAENES (rama4):    122

Secciones:
    cod_rama letra                                        descripcion
0          1     A       Agricultura, ganadería, silvicultura y pesca
1          2     B                    Explotación de minas y canteras
2          3     C                          Industrias manufactureras
3          4     D  Suministro de electricidad, gas, vapor y aire ...
4          5     E  Suministro de agua; evacuación de aguas residu...
5          6     F                                       Construcción
6          7     G  Comercio al por mayor y al p

## Tabla Rama_economica

In [24]:
df_rama = (
    df_caenes_seccion[["cod_rama", "descripcion"]]
    .rename(columns={"cod_rama": "codigo"})
    .copy()
)

faltantes = set(df_casen["rama1"].dropna().astype(int).unique()) - set(df_rama["codigo"])
assert not faltantes, f"Códigos de rama1 no encontrados en Anexo 2: {faltantes}"

print(f"Ramas económicas: {len(df_rama)}")
df_rama

Ramas económicas: 21


,codigo,descripcion
0,1,"Agricultura, ganadería, silvicultura y pesca"
1,2,Explotación de minas y canteras
2,3,Industrias manufactureras
3,4,"Suministro de electricidad, gas, vapor y aire ..."
4,5,Suministro de agua; evacuación de aguas residu...
5,6,Construcción
6,7,Comercio al por mayor y al por menor; reparaci...
7,8,Transporte y almacenamiento
8,9,Actividades de alojamiento y de servicio de co...
9,10,Información y comunicaciones


## Tabla Subrama_economica

In [25]:
# Jerarquía según los datos
df_jerarquia_datos = (
    df_casen[["rama4", "rama1"]]
    .dropna()
    .drop_duplicates()
    .astype(int)
    .rename(columns={"rama4": "codigo", "rama1": "cod_rama_datos"})
)

# Jerarquía + descripción según el anexo
df_subrama = (
    df_jerarquia_datos
    .merge(df_caenes_clase[["codigo", "cod_rama", "descripcion"]], on="codigo", how="left")
    .sort_values(["cod_rama_datos", "codigo"])
    .reset_index(drop=True)
)

# Validación 1: todas las subramas presentes en datos están en el anexo
sin_match = df_subrama[df_subrama["descripcion"].isna()]
if len(sin_match) > 0:
    print("Subramas en datos no encontradas en Anexo 2:")
    print(sin_match.head(20))
assert len(sin_match) == 0, f"{len(sin_match)} subramas en datos faltan en Anexo 2"

# Validación 2: el mapeo letra->número es consistente con los datos
inconsistencias = df_subrama[df_subrama["cod_rama"] != df_subrama["cod_rama_datos"]]
if len(inconsistencias) > 0:
    print("\n¡INCONSISTENCIAS en mapeo letra->número!")
    print("Ejemplos:")
    print(inconsistencias.head(10))
    print("\nEsto significa que A=1, B=2, ... no es correcto.")
    print("Hay que ajustar el mapeo manualmente.")
assert len(inconsistencias) == 0, "Mapeo letra->número incorrecto"

# Si pasó las validaciones, nos quedamos con la estructura final
df_subrama = (
    df_subrama[["codigo", "cod_rama", "descripcion"]]
    .rename(columns={"cod_rama": "codigo_rama"})
    .reset_index(drop=True)
)

assert df_subrama["codigo"].is_unique, "Una subrama mapea a múltiples ramas"

print(f"\nSubramas económicas: {len(df_subrama)}")
df_subrama.head(10)


Subramas económicas: 121


,codigo,codigo_rama,descripcion
0,101,1,Cultivo de plantas no perennes
1,102,1,Cultivo de plantas perennes
2,103,1,Ganadería
3,199,1,Otras actividades relacionadas con la agricult...
4,201,1,"Silvicultura, extracción de madera y actividad..."
5,301,1,"Pesca, acuicultura y actividades de servicios..."
6,401,2,Extracción y procesamiento de cobre
7,501,2,Extracción de carbón de piedra y lignito
8,601,2,Extracción de petróleo crudo y gas natural
9,701,2,"Extracción de minerales metalíferos, excepto c..."


## Tabla Hogar

In [26]:
# Validación: los atributos de hogar deben ser constantes dentro de cada folio
cols_hogar = ["area", "expr", "expp", "expc", "comuna"]
inconsistencias = df_casen.groupby("folio")[cols_hogar].nunique().gt(1).sum()
print("Inconsistencias por columna (debe ser 0 en todas):")
print(inconsistencias)
assert (inconsistencias == 0).all(), "Hay atributos de hogar que varían dentro de un folio"

df_hogar = (
    df_casen[["folio"] + cols_hogar]
    .drop_duplicates(subset="folio")
    .sort_values("folio")
    .rename(columns={"comuna": "cod_comuna"})
    .reset_index(drop=True)
)

print(f"\nHogares: {len(df_hogar):,}")
df_hogar.head()

Inconsistencias por columna (debe ser 0 en todas):
area      0
expr      0
expp      0
expc      0
comuna    0
dtype: int64

Hogares: 78,654


,folio,area,expr,expp,expc,cod_comuna
0,100020301,1,264,285,371,13127
1,100020401,1,295,286,371,13127
2,100020501,1,373,285,371,13127
3,100020801,1,266,285,372,13127
4,100070201,1,72,60,82,2203


## Tabla Persona

In [27]:
df_persona = (
    df_casen[[
        "folio", "id_persona",
        "sexo", "edad", "activ",
        "e6a", "cinef13_subarea", "o15", "rama4",
        "yoprcor",
    ]]
    .rename(columns={
        "e6a": "codigo_nivel",
        "cinef13_subarea": "codigo_subcampo",
        "o15": "codigo_cat_ocup",
        "rama4": "codigo_subrama",
    })
    .reset_index(drop=True)
)

print(f"Personas: {len(df_persona):,}")
print(f"Hogares distintos en Persona: {df_persona['folio'].nunique():,}")
df_persona.head()

Personas: 218,367
Hogares distintos en Persona: 78,654


,folio,id_persona,sexo,edad,activ,codigo_nivel,codigo_subcampo,codigo_cat_ocup,codigo_subrama,yoprcor
0,100020301,1,1,62,3.0,9.0,NaN,NaN,NaN,NaN
1,100020301,2,2,92,3.0,7.0,NaN,NaN,NaN,NaN
2,100020401,1,1,61,1.0,7.0,NaN,5.0,4805.0,1200000.0
3,100020401,2,2,61,1.0,7.0,NaN,5.0,6801.0,500000.0
4,100020401,3,1,32,1.0,12.0,5.0,5.0,4808.0,700000.0


## Verificación de integridad referencial

In [28]:
def verificar_fk(df_origen, col_fk, df_destino, col_pk, nombre):
    """Verifica que todos los valores no nulos de col_fk existen en col_pk."""
    valores_fk = set(df_origen[col_fk].dropna().unique())
    valores_pk = set(df_destino[col_pk].unique())
    huerfanos = valores_fk - valores_pk
    estado = "OK" if not huerfanos else f"FALLA ({len(huerfanos)} huérfanos)"
    print(f"  {nombre:50s} {estado}")
    return not huerfanos

print("Integridad referencial:")
verificar_fk(df_provincia, "cod_region",       df_region,     "cod_region",     "Provincia.cod_region -> Region")
verificar_fk(df_comuna,    "cod_provincia",   df_provincia,  "cod_provincia",  "Comuna.cod_provincia -> Provincia")
verificar_fk(df_subcampo,  "codigo_campo",    df_campo,      "codigo",         "Subcampo.codigo_campo -> Campo_estudio")
verificar_fk(df_subrama,   "codigo_rama",     df_rama,       "codigo",         "Subrama.codigo_rama -> Rama_economica")
verificar_fk(df_hogar,     "cod_comuna",      df_comuna,     "cod_comuna",     "Hogar.cod_comuna -> Comuna")
verificar_fk(df_persona,   "folio",           df_hogar,      "folio",          "Persona.folio -> Hogar")
verificar_fk(df_persona,   "codigo_nivel",    df_nivel_educ, "codigo",         "Persona.codigo_nivel -> Nivel_educacional")
verificar_fk(df_persona,   "codigo_subcampo", df_subcampo,   "codigo",         "Persona.codigo_subcampo -> Subcampo_estudio")
verificar_fk(df_persona,   "codigo_cat_ocup", df_cat_ocup,   "codigo",         "Persona.codigo_cat_ocup -> Categoria_ocupacional")
verificar_fk(df_persona,   "codigo_subrama",  df_subrama,    "codigo",         "Persona.codigo_subrama -> Subrama_economica")

Integridad referencial:
  Provincia.cod_region -> Region                     OK
  Comuna.cod_provincia -> Provincia                  OK
  Subcampo.codigo_campo -> Campo_estudio             OK
  Subrama.codigo_rama -> Rama_economica              OK
  Hogar.cod_comuna -> Comuna                         OK
  Persona.folio -> Hogar                             OK
  Persona.codigo_nivel -> Nivel_educacional          OK
  Persona.codigo_subcampo -> Subcampo_estudio        OK
  Persona.codigo_cat_ocup -> Categoria_ocupacional   OK
  Persona.codigo_subrama -> Subrama_economica        OK


True

# Parte 3: Exportación de cada tabla a CSV

## Exportar tablas a CSV

In [29]:
import os

DIR_TABLAS = "casen/tablas"
os.makedirs(DIR_TABLAS, exist_ok=True)

TABLAS = {
    "region":                df_region,
    "provincia":             df_provincia,
    "comuna":                df_comuna,
    "nivel_educacional":     df_nivel_educ,
    "campo_estudio":         df_campo,
    "subcampo_estudio":      df_subcampo,
    "categoria_ocupacional": df_cat_ocup,
    "rama_economica":        df_rama,
    "subrama_economica":     df_subrama,
    "hogar":                 df_hogar,
    "persona":               df_persona,
}

print(f"Exportando a {DIR_TABLAS}/")
for nombre, df in TABLAS.items():
    ruta = os.path.join(DIR_TABLAS, f"{nombre}.csv")
    df.to_csv(ruta, index=False)
    print(f"  {nombre:25s} {len(df):>7,} filas  ->  {ruta}")

Exportando a casen/tablas/
  region                         16 filas  ->  casen/tablas/region.csv
  provincia                      56 filas  ->  casen/tablas/provincia.csv
  comuna                        346 filas  ->  casen/tablas/comuna.csv
  nivel_educacional              15 filas  ->  casen/tablas/nivel_educacional.csv
  campo_estudio                  11 filas  ->  casen/tablas/campo_estudio.csv
  subcampo_estudio               27 filas  ->  casen/tablas/subcampo_estudio.csv
  categoria_ocupacional           9 filas  ->  casen/tablas/categoria_ocupacional.csv
  rama_economica                 21 filas  ->  casen/tablas/rama_economica.csv
  subrama_economica             121 filas  ->  casen/tablas/subrama_economica.csv
  hogar                      78,654 filas  ->  casen/tablas/hogar.csv
  persona                   218,367 filas  ->  casen/tablas/persona.csv


In [30]:
print("Verificación de CSV exportados:")
for nombre, df in TABLAS.items():
    ruta = os.path.join(DIR_TABLAS, f"{nombre}.csv")
    df_leido = pd.read_csv(ruta)
    ok_filas = df_leido.shape[0] == df.shape[0]
    ok_cols = list(df_leido.columns) == list(df.columns)
    estado = "OK" if (ok_filas and ok_cols) else "FALLA"
    print(f"  {nombre:25s} {estado:6s} ({df_leido.shape[0]:>7,} filas, {df_leido.shape[1]} columnas)")
    assert ok_filas, f"Diferencia de filas en {nombre}"
    assert ok_cols,  f"Diferencia de columnas en {nombre}"

Verificación de CSV exportados:
  region                    OK     (     16 filas, 2 columnas)
  provincia                 OK     (     56 filas, 3 columnas)
  comuna                    OK     (    346 filas, 3 columnas)
  nivel_educacional         OK     (     15 filas, 2 columnas)
  campo_estudio             OK     (     11 filas, 2 columnas)
  subcampo_estudio          OK     (     27 filas, 3 columnas)
  categoria_ocupacional     OK     (      9 filas, 2 columnas)
  rama_economica            OK     (     21 filas, 2 columnas)
  subrama_economica         OK     (    121 filas, 3 columnas)
  hogar                     OK     ( 78,654 filas, 6 columnas)
  persona                   OK     (218,367 filas, 10 columnas)
